##### Part 5 — Build gold reporting outputs
Referring to code snippets from Python program: 
[notebooks/04_build_gold_reports.py].

[1] - Use **"Layered SQL"** to create business-facing reporting outputs from the silver dataset. 

NOTE: Foundation is 1st "Temporary View" on the silver dataset.

[1.1] - Create UNITY CATALOG tables: The path to SILVER LAYER

In [0]:
# SETTING CONSTANTS ACCORDING TO:
# workspace.prj_fintech-transaction-reporting-on-databricks-with-spark_sc/
catalog_name = "workspace"
schema_name = "prj_fintech-transaction-reporting-on-databricks-with-spark_sc"

silver_table = f"{catalog_name}.`{schema_name}`.silver_transactions"
print("Silver Table incl. PATH:", silver_table)

[1.2] - Create TEMPORARY VIEW: on SILVER LAYER table

In [0]:
# LEARNING Template: 
#
# (1) Creating a TEMP VIEW on a UNITY CATALOG -> "TABLE" has this SYNTAX: 
# 
# spark.table(<table name incl entire path to UNITY CATALOG>).createOrReplaceTempView("<view name>")
# 
# (2) Creating a TEMP VIEW on a UNITY CATALOG -> "FILE" has this SYNTAX:
# 
# <DataFrame name>.createOrReplaceTempView("<view name>")", 
# !! => where the DataFrame already was created and loaded from file.

In [0]:
spark.table(silver_table).createOrReplaceTempView("silver_transactions_view")

display(spark.sql("SELECT * FROM silver_transactions_view LIMIT 10"))

### REWORK: Document the below steps with additional Text-Cells FROM HERE !

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW focused_transactions AS
SELECT
    transaction_id,
    account_id,
    customer_id,
    report_date,
    transaction_type,
    amount,
    transaction_status,
    payment_channel,
    merchant_country,
    amount_band,
    is_high_value,
    is_international
FROM silver_transactions_view
""")

display(spark.sql("SELECT * FROM focused_transactions LIMIT 10"))

In [0]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW daily_summary_view AS
SELECT
    report_date,
    COUNT(*) AS total_transactions,
    ROUND(SUM(amount), 2) AS total_amount,
    SUM(CASE WHEN transaction_status = 'SUCCESSFUL' THEN 1 ELSE 0 END) AS successful_transactions,
    SUM(CASE WHEN transaction_status = 'FAILED' THEN 1 ELSE 0 END) AS failed_transactions,
    ROUND(AVG(amount), 2) AS avg_amount
FROM focused_transactions
GROUP BY report_date
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW failed_summary_view AS
SELECT
    report_date,
    transaction_type,
    payment_channel,
    COUNT(*) AS failed_count,
    ROUND(SUM(amount), 2) AS failed_amount
FROM focused_transactions
WHERE transaction_status = 'FAILED'
GROUP BY report_date, transaction_type, payment_channel
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW payment_channel_summary_view AS
SELECT
    report_date,
    payment_channel,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS avg_amount
FROM focused_transactions
GROUP BY report_date, payment_channel
""")

In [0]:
# TODO:
# Create two more temp views:
# 1. high_value_transfers_view
# 2. international_summary_view

spark.sql("""
CREATE OR REPLACE TEMP VIEW high_value_transfers_view AS
SELECT
    report_date,
    account_id,
    customer_id,
    transaction_id,
    amount,
    transaction_type,
    payment_channel,
    merchant_country
FROM focused_transactions
WHERE is_high_value = 1
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW international_summary_view AS
SELECT
    report_date,
    merchant_country,
    COUNT(*) AS transaction_count,
    ROUND(SUM(amount), 2) AS total_amount
FROM focused_transactions
WHERE is_international = 1
GROUP BY report_date, merchant_country
""")

### REWORK: Test Out 2 things: 
#### (1) Delete table from UC programmatically (see below cell). 
##### (1.0) note down the behaviour of cell-execution and result in UC table.

#### (2) Keep one of the other already Created & Filled tables in UC, but via the table`s feeding view above, 
##### (2.1) ADD in one table a Column
##### (2.2) REMOVE in another table a Column 
##### (2.0) note down the behaviour after the according below table "CREATE OR REPLACE" execution !

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {catalog_name}.`{schema_name}`.gold_daily_summary")

In [0]:
# Save the final gold tables
spark.sql(f"CREATE OR REPLACE TABLE {catalog_name}.`{schema_name}`.gold_daily_summary AS SELECT * FROM daily_summary_view")
spark.sql(f"CREATE OR REPLACE TABLE {catalog_name}.`{schema_name}`.gold_failed_transactions AS SELECT * FROM failed_summary_view")
spark.sql(f"CREATE OR REPLACE TABLE {catalog_name}.`{schema_name}`.gold_high_value_transfers AS SELECT * FROM high_value_transfers_view")
spark.sql(f"CREATE OR REPLACE TABLE {catalog_name}.`{schema_name}`.gold_payment_channel_summary AS SELECT * FROM payment_channel_summary_view")
spark.sql(f"CREATE OR REPLACE TABLE {catalog_name}.`{schema_name}`.gold_international_summary AS SELECT * FROM international_summary_view")

In [0]:
display(spark.table(f"{catalog_name}.`{schema_name}`.gold_daily_summary"))
display(spark.table(f"{catalog_name}.`{schema_name}`.gold_failed_transactions"))
display(spark.table(f"{catalog_name}.`{schema_name}`.gold_high_value_transfers"))
display(spark.table(f"{catalog_name}.`{schema_name}`.gold_payment_channel_summary"))
display(spark.table(f"{catalog_name}.`{schema_name}`.gold_international_summary"))
